# AI Incidents Pipeline Runner

Loads the configuration from `config/params.yaml` and runs the full pipeline:
ingestion -> preprocessing -> NLP -> sentiment -> analysis -> visualization/report.

In [1]:
from pathlib import Path

import yaml

from src import configure_logging, set_global_seeds
from src.ingestion import load_and_validate
from src.preprocessing import preprocess
from src.nlp import process_text
from src.sentiment import run_sentiment
from src.analysis import run_analysis
from src.visualization import run_visualization
from src.html_report import generate_html_report

In [2]:
with open("config/params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

configure_logging()
set_global_seeds(config["random_state"])

In [3]:
df = load_and_validate(config)
df.head()

2026-08-08 22:36:34,507 [INFO] src.ingestion: Loaded 10372 rows from data\raw\oecd_aim.csv (encoding=utf-8)
2026-08-08 22:36:34,523 [INFO] src.ingestion: Column 'text_data' has 0.9% null values
2026-08-08 22:36:34,573 [INFO] src.ingestion: No duplicate rows found


,id,title,text_data,evidences,country,tags_list,industries,harmed,harmlevel,harmtype,threat,event_date,geo_zone
0,4,Clues to Future Snowden Leaks Found In His Past,Only a tiny fraction of Snowden's documents ha...,['A new article by investigative reporter Chri...,USA,"['Respect of human rights', 'Transparency & ex...","['Digital security', 'Government, security and...",['General public'],[],['Psychological'],True,2014-01-02,North America
1,7,"'Talking Angela' programmer talks hoaxes, AI m...","If you're not a parent or teenager, it's possi...","['In fact, Talking Angela is a hugely popular ...",NaN,"['Respect of human rights', 'Robustness & digi...",['Digital security'],['Unknown'],['Non-physical harm'],['Reputational'],False,2014-03-01,Unknown
2,10,Technology: Rise of the replicants,Rapid advances in artificial intelligence now ...,['Rapid advances in artificial intelligence no...,USA,"['Performance', 'Reskill or upskill']","['Financial and insurance services', 'IT infra...",['Workers'],[],['Economic/Property'],True,2014-03-04,North America
3,18,"WeChat, Microsoft trade fire over 'killing' of...",China's popular mobile messaging application W...,"[""WeChat, Microsoft trade fire over 'killing' ...",CHN,"['Robustness & digital security', 'Privacy & d...","['IT infrastructure and hosting', 'Robots, sen...",['Unknown'],['Non-physical harm'],['Reputational'],False,2014-06-02,Asia
4,22,Is the Turing Test milestone all it's cracked ...,Horrific accident kills Utah toddler on visit ...,['A computer program mimicked a conversation a...,USA,"['Transparency & explainability', 'Robustness ...","['IT infrastructure and hosting', 'Robots, sen...",[],[],[],False,2014-06-10,North America


In [4]:
df = preprocess(df, config)
df.head()

2026-08-08 22:36:34,876 [INFO] src.preprocessing: No year_range filter applied — keeping all 10372 rows
2026-08-08 22:36:34,891 [INFO] src.preprocessing: No regions filter applied — keeping all 10372 rows


,id,title,text_data,evidences,country,tags_list,industries,harmed,harmlevel,harmtype,...,mlb_industries__Food and beverages,"mlb_industries__Government, security and defence","mlb_industries__Healthcare, drugs and biotechnology",mlb_industries__IT infrastructure and hosting,"mlb_industries__Logistics, wholesale and retail","mlb_industries__Media, social platforms, marketing",mlb_industries__Mobility and autonomous vehicles,mlb_industries__Real estate,"mlb_industries__Robots, sensors, IT hardware","mlb_industries__Travel, leisure and hospitality"
0,4,Clues to Future Snowden Leaks Found In His Past,Only a tiny fraction of Snowden's documents ha...,['A new article by investigative reporter Chri...,USA,"['Respect of human rights', 'Transparency & ex...","[Digital security, Government, security and de...",['General public'],[],['Psychological'],...,0,1,0,0,0,0,0,0,0,0
1,7,"'Talking Angela' programmer talks hoaxes, AI m...","If you're not a parent or teenager, it's possi...","['In fact, Talking Angela is a hugely popular ...",NaN,"['Respect of human rights', 'Robustness & digi...",[Digital security],['Unknown'],['Non-physical harm'],['Reputational'],...,0,0,0,0,0,0,0,0,0,0
2,10,Technology: Rise of the replicants,Rapid advances in artificial intelligence now ...,['Rapid advances in artificial intelligence no...,USA,"['Performance', 'Reskill or upskill']","[Financial and insurance services, IT infrastr...",['Workers'],[],['Economic/Property'],...,0,0,0,1,0,0,0,0,1,0
3,18,"WeChat, Microsoft trade fire over 'killing' of...",China's popular mobile messaging application W...,"[""WeChat, Microsoft trade fire over 'killing' ...",CHN,"['Robustness & digital security', 'Privacy & d...","[IT infrastructure and hosting, Robots, sensor...",['Unknown'],['Non-physical harm'],['Reputational'],...,0,0,0,1,0,0,0,0,1,0
4,22,Is the Turing Test milestone all it's cracked ...,Horrific accident kills Utah toddler on visit ...,['A computer program mimicked a conversation a...,USA,"['Transparency & explainability', 'Robustness ...","[IT infrastructure and hosting, Robots, sensor...",[],[],[],...,0,0,0,1,0,0,0,0,1,0


In [5]:
df = process_text(df, config)
df[["tokens", "mental_health_flag"]].head()

2026-08-08 22:36:35,009 [INFO] src.nlp: Downloading NLTK resource 'wordnet'
2026-08-08 22:36:35,358 [INFO] src.nlp: Downloading NLTK resource 'omw-1.4'


,tokens,mental_health_flag
0,"[tiny, fraction, snowdens, document, published...",0
1,"[youre, parent, teenager, possible, first, hea...",0
2,"[rapid, advance, artificial, intelligence, thr...",0
3,"[china, popular, mobile, messaging, applicatio...",0
4,"[horrific, accident, kill, utah, toddler, visi...",0


In [6]:
df = run_sentiment(df, config)
if "sentiment_score" in df.columns:
    display(df[["sentiment_score", "sentiment_label"]].head())
else:
    print("Sentimiento desactivado.")

c:\Users\USER\Documents\ucom\tesis\incidents-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
De

KeyboardInterrupt: 

In [ ]:
processed_path = Path(config["data"]["processed_path"])
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path)

In [ ]:
metrics = run_analysis(df, config)

2026-08-08 12:34:43,528 [INFO] src.analysis: Metrics written to outputs\reports\metrics.json


In [ ]:
html_path = generate_html_report(df, metrics, config)
print(f"Reporte interactivo → {html_path}")

2026-08-08 12:34:43,880 [INFO] src.html_report: Interactive HTML report → outputs\reports\interactive_report.html


Reporte interactivo → outputs\reports\interactive_report.html


In [ ]:
print("Pipeline completado.")

Pipeline completado.
